In [ ]:
## Retrieval augmented generation with LlamaIndex
# Put your PDFs / text files in the "data" folder before running.
%pip install -q -U llama-index-core llama-index-llms-openai llama-index-embeddings-openai llama-index-readers-file python-dotenv

import os
from dotenv import load_dotenv
load_dotenv()


In [ ]:
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to your .env file"

# LlamaIndex defaults to old OpenAI models, so set current ones explicitly.
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAI(model=os.getenv("OPENAI_MODEL", "gpt-6-luna"))
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")


In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
documents = SimpleDirectoryReader("data").load_data()


In [ ]:
documents

In [5]:
index=VectorStoreIndex.from_documents(documents,show_progress=True)

e:\New Recordings\Langchain Videos\LLAmindex\Projects\Llama_index\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating embeddings: 100%|██████████| 36/36 [00:03<00:00, 11.11it/s]


In [6]:
index

In [7]:
query_engine=index.as_query_engine()

In [ ]:
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor

retriever = VectorIndexRetriever(index=index, similarity_top_k=4)
postprocessor = SimilarityPostprocessor(similarity_cutoff=0.50)

query_engine = RetrieverQueryEngine(retriever=retriever,
                                    node_postprocessors=[postprocessor])


In [ ]:
response = query_engine.query("What is attention is all you need?")


In [ ]:
from llama_index.core.response.pprint_utils import pprint_response
pprint_response(response, show_source=True)
print(response)


In [ ]:
import os.path
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

# check if storage already exists
PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    documents = SimpleDirectoryReader("data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

# either way we can now query the index
query_engine = index.as_query_engine()
response = query_engine.query("What are transformers?")
print(response)